In [ ]:
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/liya_diploma'
    sys.path.insert(0, DRIVE_ROOT)
    # FLUX training via Hugging Face's official example script — no ai-toolkit,
    # no numpy/scipy version war. Install only what the script needs.
    get_ipython().system(f'pip install -q -r {DRIVE_ROOT}/requirements_aitoolkit_flux.txt')

    # Pull the latest train_dreambooth_lora_flux.py from the diffusers repo.
    # It's a single self-contained script — easier than vendoring it.
    DIFFUSERS_VERSION = 'main'  # use 'v0.32.0' or similar tag if you want to pin
    SCRIPT_URL = (
        f'https://raw.githubusercontent.com/huggingface/diffusers/{DIFFUSERS_VERSION}'
        '/examples/dreambooth/train_dreambooth_lora_flux.py'
    )
    get_ipython().system(f'wget -q -O /content/train_dreambooth_lora_flux.py {SCRIPT_URL}')
    print(f"Downloaded training script → /content/train_dreambooth_lora_flux.py")
except ModuleNotFoundError:
    # Local run: find project root by looking for scripts/ directory
    _here = Path().resolve()
    DRIVE_ROOT = str(_here if (_here / 'scripts').exists() else _here.parent)
    sys.path.insert(0, DRIVE_ROOT)
    print("Local run — make sure diffusers/transformers/accelerate are installed in your venv.")

print(f"DRIVE_ROOT: {DRIVE_ROOT}")

In [ ]:
!sed -i 's/^check_min_version/# check_min_version/' /content/train_dreambooth_lora_flux.py
!grep "check_min_version" /content/train_dreambooth_lora_flux.py

In [ ]:
import os, sys, runpy
from pathlib import Path

# Reuse train_images prepared by notebook 04 cell-02 (or recreate here).
IS_COLAB = DRIVE_ROOT.startswith('/content/drive')
TRAIN_IMAGES = '/content/data/train_images' if IS_COLAB else f'{DRIVE_ROOT}/data/train_images'

# If the folder doesn't exist (e.g. fresh Colab runtime), regenerate it from train.jsonl.
if not Path(TRAIN_IMAGES).exists() or not list(Path(TRAIN_IMAGES).glob('*.png')):
    import json, shutil
    from tqdm.auto import tqdm
    Path(TRAIN_IMAGES).mkdir(parents=True, exist_ok=True)
    with open(f'{DRIVE_ROOT}/data/train.jsonl', encoding='utf-8') as f:
        pairs = [json.loads(l) for l in f]
    for item in tqdm(pairs, desc='Copy train images'):
        stem = Path(item['png_path']).stem
        shutil.copy(item['png_path'], f'{TRAIN_IMAGES}/{stem}.png')
        cap = item['caption']
        if not cap.startswith('LOGOIMG'):
            cap = f'LOGOIMG {cap}'
        Path(f'{TRAIN_IMAGES}/{stem}.txt').write_text(cap, encoding='utf-8')
    print(f"Prepared {len(pairs)} pairs → {TRAIN_IMAGES}")

OUTPUT_DIR = f'{DRIVE_ROOT}/results/experiments/flux_r16'
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

SCRIPT = '/content/train_dreambooth_lora_flux.py'
if not Path(SCRIPT).exists() or Path(SCRIPT).stat().st_size < 5000:
    raise RuntimeError(
        f"{SCRIPT} missing or truncated ({Path(SCRIPT).stat().st_size if Path(SCRIPT).exists() else 0} bytes). "
        "Re-run cell-1-setup."
    )

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

# Inline run: replace sys.argv and execute the script as __main__. Any error
# becomes a normal Python traceback in this cell — no hidden stderr.
ARGV = [
    SCRIPT,
    '--pretrained_model_name_or_path=black-forest-labs/FLUX.1-dev',
    f'--instance_data_dir={TRAIN_IMAGES}',
    f'--output_dir={OUTPUT_DIR}',
    '--instance_prompt=LOGOIMG',
    '--mixed_precision=bf16',
    '--resolution=512',
    '--train_batch_size=1',
    '--gradient_accumulation_steps=4',
    '--gradient_checkpointing',
    '--use_8bit_adam',
    '--learning_rate=1e-4',
    '--lr_scheduler=cosine',
    '--lr_warmup_steps=100',
    '--max_train_steps=4000',
    '--checkpointing_steps=500',
    '--rank=16',
    '--seed=42',
    '--cache_latents',
]

print("Launching FLUX.1-dev LoRA training (rank=16, 4000 steps)...")
print("Args:", ' '.join(ARGV[1:]))

_old_argv = sys.argv
try:
    sys.argv = ARGV
    runpy.run_path(SCRIPT, run_name='__main__')
finally:
    sys.argv = _old_argv

print("\nDONE")

Launching FLUX.1-dev LoRA training (rank=16, 4000 steps)...
Args: --pretrained_model_name_or_path=black-forest-labs/FLUX.1-dev --instance_data_dir=/content/data/train_images --output_dir=/content/drive/MyDrive/liya_diploma/results/experiments/flux_r16 --instance_prompt=LOGOIMG --mixed_precision=bf16 --resolution=512 --train_batch_size=1 --gradient_accumulation_steps=4 --gradient_checkpointing --use_8bit_adam --learning_rate=1e-4 --lr_scheduler=cosine --lr_warmup_steps=100 --max_train_steps=4000 --checkpointing_steps=500 --rank=16 --seed=42 --cache_latents


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
You are using a model of type clip_text_model to instantiate a model of type . This is not supported for all configurations of models and can yield errors.
You are using a model of type t5 to instantiate a model of type . This is

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

All model checkpoint weights were used when initializing AutoencoderKL.

All the weights of AutoencoderKL were initialized from the model checkpoint at black-forest-labs/FLUX.1-dev.
If your task is similar to the task the model of the checkpoint was trained on, you can already use AutoencoderKL for predictions without further training.


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

{'out_channels', 'axes_dims_rope'} was not found in config. Values will be initialized to default values.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
from diffusers import StableDiffusionPipeline
import torch
from pathlib import Path

TEST_PROMPTS = [
    "minimalist coffee shop logo, circular icon, dark green, flat vector design, white background",
    "tech startup logo, geometric hexagon, blue gradient, bold sans-serif, white background",
    "bakery logo, wheat sheaf icon, warm brown, handcrafted artisan style, white background",
    "fitness brand, lion silhouette, orange and black, bold geometric, white background",
    "law firm logo, balanced scales, navy blue, serif elegant typography, white background",
]

out_dir = f'{DRIVE_ROOT}/results/experiments/sd15_baseline'
Path(out_dir).mkdir(parents=True, exist_ok=True)

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
).to("cuda")
pipe.set_progress_bar_config(disable=True)

for i, prompt in enumerate(TEST_PROMPTS):
    imgs = pipe(
        prompt,
        negative_prompt="photorealistic, blurry, cluttered, complex background, 3D render",
        num_images_per_prompt=2,
        generator=torch.Generator().manual_seed(42),
        guidance_scale=7.5,
        num_inference_steps=30,
        height=512,
        width=512,
    ).images
    for j, img in enumerate(imgs):
        img.save(f"{out_dir}/prompt{i:02d}_v{j}.png")

del pipe; torch.cuda.empty_cache()
print(f"SD 1.5 baseline: {len(TEST_PROMPTS)*2} images \u2192 {out_dir}")

In [ ]:
from diffusers import FluxPipeline
import torch

flux_lora_path = f'{DRIVE_ROOT}/results/experiments/flux_r16'
out_dir = f'{DRIVE_ROOT}/results/experiments/flux_r16_samples'
Path(out_dir).mkdir(parents=True, exist_ok=True)

FLUX_PROMPTS = ["LOGOIMG " + p.replace(", white background", "") for p in TEST_PROMPTS]

pipe = FluxPipeline.from_pretrained(
    "black-forest-labs/FLUX.1-dev",
    torch_dtype=torch.bfloat16,
).to("cuda")
pipe.load_lora_weights(flux_lora_path)
pipe.set_progress_bar_config(disable=True)

for i, prompt in enumerate(FLUX_PROMPTS):
    imgs = pipe(
        prompt,
        num_images_per_prompt=2,
        generator=torch.Generator().manual_seed(42),
        guidance_scale=3.5,
        num_inference_steps=20,
        height=512, width=512,
    ).images
    for j, img in enumerate(imgs):
        img.save(f"{out_dir}/prompt{i:02d}_v{j}.png")

del pipe; torch.cuda.empty_cache()
print(f"FLUX LoRA: {len(FLUX_PROMPTS)*2} images → {out_dir}")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

# UPDATE BEST_SDXL_RANK to the rank with lowest FID from Experiment 1 (notebook 07)
BEST_SDXL_RANK = 16

MODELS = {
    "SD 1.5 Baseline": f'{DRIVE_ROOT}/results/experiments/sd15_baseline',
    f"SDXL LoRA r={BEST_SDXL_RANK}": f'{DRIVE_ROOT}/results/experiments/sdxl_r{BEST_SDXL_RANK}_samples',
    "FLUX.1-dev LoRA": f'{DRIVE_ROOT}/results/experiments/flux_r16_samples',
}

fig, axes = plt.subplots(5, 3, figsize=(12, 20))
for row in range(5):
    for col, (model_name, img_dir) in enumerate(MODELS.items()):
        img_path = f"{img_dir}/prompt{row:02d}_v0.png"
        if Path(img_path).exists():
            axes[row, col].imshow(Image.open(img_path))
        else:
            axes[row, col].text(0.5, 0.5, "Not yet\ngenerated",
                                ha='center', va='center', fontsize=8)
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(model_name, fontsize=10, fontweight='bold')

plt.suptitle("Experiment 3: SD1.5 Baseline vs SDXL LoRA vs FLUX LoRA", fontsize=12)
plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/results/experiments/exp3_model_comparison.png', dpi=150)
plt.show()